# Dexwin Pay — live assessment notebook

Read `README.md` and `data/DATA_CARD.md` first.

Goal: **catch fraud, cut false declines.** The cells below are a weak baseline. Improve them, then be ready to talk serving and monitoring.

Ignore `INTERVIEWER.md`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DATA_PATH = next(
    p for p in (Path("data/transactions.csv"), Path("../data/transactions.csv"))
    if p.exists()
)
df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

,txn_id,event_ts,amount,currency,merchant_id,merchant_category,channel,country,customer_id,customer_tenure_days,txns_30d,avg_amount_30d,prior_declines_30d,hour,is_new_device,payment_method,device_fingerprint,ops_review_code,rules_engine_decline,is_fraud
0,TXN-000790,19/08/2026 22:38,9.42,NGN,M-2017,NaN,ios,NGA,C-10165,548.0,11,9.50,0,1.0,1,momo,DEV-45366,CNF,Y,1
1,TXN-002393,2026-08-09T11:21:00,32.23,GHS,M-2048,restaurants,ios,gh,C-10578,1.0,11,18.70,0,11.0,0,momo,DEV-27711,CLR,N,0
2,TXN-000145,03/09/2026 23:54,34.02,USD,M-2073,electronics,android,Kenya,C-10623,1442.0,8,49.68,1,1.0,0,card,DEV-14008,CLR,N,0
3,TXN-001134,05/09/2026 05:33,67.43,GHS,M-2041,NaN,android,ng,C-10283,720.0,34,88.69,0,12.0,0,card,DEV-83974,CLR,N,0
4,TXN-000464,17/08/2026 14:38,8.67,GHS,M-2016,RESTAURANTS,web,Ghana,C-10564,448.0,0,12.06,5,11.0,0,card,DEV-59655,CLR,Y,0


In [2]:
print(df_raw.dtypes)
print()
print(df_raw["is_fraud"].value_counts(normalize=True))

txn_id                   object
event_ts                 object
amount                   object
currency                 object
merchant_id              object
merchant_category        object
channel                  object
country                  object
customer_id              object
customer_tenure_days    float64
txns_30d                  int64
avg_amount_30d          float64
prior_declines_30d        int64
hour                    float64
is_new_device             int64
payment_method           object
device_fingerprint       object
ops_review_code          object
rules_engine_decline     object
is_fraud                  int64
dtype: object

is_fraud
0    0.915475
1    0.084525
Name: proportion, dtype: float64


## Weak baseline

This matches `scripts/baseline.py`. It is a starting point, not a target design.

In [3]:
df = df_raw.copy()
n_loaded = len(df)

df["amount"] = (
    df["amount"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
    .replace("", pd.NA)
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

df = df.dropna().copy()

y = df["is_fraud"]
X = df.drop(columns=["is_fraud"])

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=250)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"rows loaded:       {n_loaded}")
print(f"rows after dropna: {len(df)}")
print(f"accuracy:          {accuracy_score(y_test, pred):.4f}")
print(confusion_matrix(y_test, pred))
# TODO: product asked for fewer false declines — add that later.

rows loaded:       3218
rows after dropna: 2368
accuracy:          0.9262
[[439   0]
 [ 35   0]]


/workspaces/dexwin-ml-assessment/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/workspaces/dexwin-ml-assessment/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Your work

Clean the table, choose features that would exist at authorization time, pick a metric that matches the product, and train something you would consider shadowing.

Do not spend the whole session on hyperparameter search.

In [ ]:
# Your cleaning / features / model.
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer

COUNTRY_MAP = {"gh": "GH", "GHA": "GH", "Ghana": "GH",
               "ng": "NG", "nga": "NG", "Nigeria": "NG",
               "ke": "KE", "ken": "KE", "Kenya": "KE",}

def parse_event_ts(s: pd.Series) -> tuple[pd.Series, pd.Series]:
    s = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype= "datetime64[ns]")
    iso= s.str.match(r"^\d{4}-- ")
    slash= s.str.match( r"^\d{2}-\d{2}/\d{4}")
    date_only = s.str.match(r"^\d{2}-\d{2}/\d{4}")
    out[iso] = pd.to_datetime(s[iso], format="ISO8601")
    out[slash] = pd.to_datetime(s[slash], format= "%d%m%y %H:%M")
    out[date_only] =pd.to_datetime(s[date_only], format= "%m-%d-%Y")
    return out, ~date_only

def clean( raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.drop_duplicates().drop_duplicates("txn_id")
    df["amount"] = pd.to_numeric(
        df["amount"].astype(str).str.replace(r"[^0-9]", "", regex=True).replace("", np.nan), errors= "coerce",)
    df["events_ts"], has_time= parse_event_ts(df["event_ts"])
    df ["hour"] = df["hour"].fillna(df["event_ts"].dt.hour.where(has_time))
    df ["country"] = df["country"].str.strip().str.lower().map(COUNTRY_MAP)

    

df = clean(df_raw)


def build_features(df: pd.DataFrame) ->pd.DataFrame:
    Table=pd.DataFrame(idex=df.index)
    Table["log_amount"] = np.log1p(df["amount"])
    Table["log_amount_ratio"] = np.log1p(df["amount"] / df["avg_amount_30d"])
    Table["log_tenure"] = np.log1p(df["customer_tenure_days"])
    Table["log_txn_30d"] = np.log1p(df["txn_30d"])
    Table["prior_declines_30d"] = df["prior_declines_30d"]
    Table["is_new_device"] = df["is_new_device"]
    Table["is_night"] = df["hour"].between(0,4).astype(float).where(df["hour"].notna())

    for c in CAT +["merchant_id"]:
        Table[c] = df[c.astype(str)]
    return Table

Table = build_features(df)
y = df["is_fraud"].to_numpy()
rules = (df["rules_engine_decline"] == "Y").to_numpy()


n = len(df)
i_val, i_test = int(n * 0.7), int(n *0.8)
train, val, test = slice( 0, i_val), slice(i_val, i_test), slice(i_test, n)

for name part in [("train")]

    




    


AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
# Metrics you would actually report to the product owner.


## Serving and monitoring (write bullets, then talk)

Freeze the model around here and answer in words:

1. What calls the model, and what is in the request?
2. Which features are **not** safe to serve online from this warehouse dump?
3. How do you roll it out without instantly changing every merchant’s decline rate?
4. Fraud labels arrive late. What do you monitor **tomorrow morning** vs next week?
5. How do you notice leakage / feedback loops once the model is live?

In [ ]:
# Optional notes for the discussion (not graded as code).
